## 0. Init libraries en helper functions

In [461]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [462]:
import pandas as pd
import re
import numpy as np

# from arcgis.gis import GIS
# from arcgis.apps import survey123

import sqlite3
from datetime import datetime

In [463]:
import os
import sys

# Get the absolute path of the folder above the notebook
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from src import utils

In [464]:
pd.options.display.max_columns = None

### 0.1 Inladen gegevens

Gebruikt nu nog exports via LSVI-package (R). Testen of export uit sqlite db kan gemaakt worden.

In [465]:
df_vereisten = pd.read_csv('../input/invoervereistenUitTeWerken-UTF8.csv', sep=';')
# We should only use vereiesten v3
df_vereisten = df_vereisten[df_vereisten['Versie'] == 'Versie 3']
# Maak unieke ID aan voor vragen adhv voorwaarde id + habitattype
df_vereisten['vraag_id'] = "vraag_" + df_vereisten['VoorwaardeID'].astype(str) + "_" + (df_vereisten['Habitatsubtype'].astype(str).apply(utils.clean_name))
df_vereisten['test_function'] = df_vereisten['Habitatsubtype'].astype(str).apply(utils.clean_name)
# Make sure vereisten ID (VoorwaardeID) is uniek
df_vereisten.drop_duplicates(subset=['vraag_id'], inplace=True)

df_soorten = pd.read_csv('../input/invoervereistenUitTeWerkenSoortenlijst-UTF8.csv', sep=';')
df_schalen = pd.read_csv('../input/survey123_schalen.csv', sep=';') # Het bestand van de vorige stap!

In [466]:
# For testing we keep survey small
# df_vereisten = df_vereisten[df_vereisten.Habitatsubtype.str[0].astype(int)<7]

In [467]:
df_vereisten.Versie.unique() 

<ArrowStringArray>
['Versie 3']
Length: 1, dtype: str

In [468]:
# df_vereisten[['AnalyseVariabele','Eenheid']].drop_duplicates()

**Komt reeds uit SQLite**

In [469]:
df_habitattypes = pd.read_sql_table('Habitattype', 'sqlite:///../input/LSVIHabitatTypes.sqlite')

In [470]:
# df_habitattypes[~df_habitattypes['Code'].str.startswith('rbb')].HabitatgroepId.unique()
# df_habitattypes[~df_habitattypes['Code'].str.startswith('rbb')].Code.unique()

**BWK-karteringseenheden: betere bron nog te gebruiken**

In [471]:
df_bwk_kart_eenh = pd.read_csv('../input/bwk_karteringseenheden.csv', sep=';')

In [472]:
df_vereisten[df_vereisten['vraag_id'] == '6410_mo']  #vraag_rbbsm_verruiging_verruiging_sublijst_2
df_vereisten.head()
# df_habitattypes[df_habitattypes['Code'] == '6410']
df_vereisten.Habitattype.unique()

<ArrowStringArray>
['1310', '1320', '1330', '2110', '2120', '2130', '2150', '2160', '2170',
 '2180', '2190', '2310', '2330', '3110', '3130', '3140', '3150', '3160',
 '3260', '3270', '4010', '4030', '5130', '6120', '6210', '6230', '6410',
 '6430', '6510', '7110', '7140', '7150', '7210', '7220', '7230', '9110',
 '9120', '9130', '9150', '9160', '9190', '91E0', '91F0']
Length: 43, dtype: str

## 1. Choices

In [473]:
print("Choices tabblad opbouwen...")
choices_list = []

# A. Schalen toevoegen
for _, row in df_schalen.iterrows():
    choices_list.append(row.to_dict())

# B. Dynamische Habitattype lijst genereren (Voor de eerste vraag)
unieke_habitats = df_vereisten['Habitattype'].dropna().unique()
for hab in unieke_habitats:
    choices_list.append({
        "list_name": "lijst_habitats",
        "name": utils.clean_name(hab),
        "label": str(hab).upper()
    })

# C. Dynamische Soortenlijsten genereren (Groepeer per TaxongroepId)
for _, soort in df_soorten.iterrows():
    if pd.notna(soort['TaxongroepId']):
        tax_id = int(soort['TaxongroepId'])
        choices_list.append({
            "list_name": f"taxa_{tax_id}",
            "name": utils.clean_name(soort['WetNaamKort']),
            "label": f"{soort['NedNaam']} ({soort['WetNaamKort']})" if pd.notna(soort['NedNaam']) else soort['WetNaamKort']
        })

# Dynamische subhapitattypes toevoegen (Voor de BWK-vragen)
unieke_subhabitats = df_vereisten['Habitatsubtype'].dropna().unique()
for hab in unieke_subhabitats:
    choices_list.append({
        "list_name": "lijst_subhabitats",
        "name": utils.clean_name(hab),
        "label": str(hab).upper()
    })

# BWK Karteringseenheden toevoegen
for _, row in df_bwk_kart_eenh.iterrows():
    choices_list.append({
        "list_name": "lijst_bwk_karteringseenheden",
        "name": utils.clean_name(row["karteringseenheid"]),
        "label": str(row['karteringseenheid'])
    })

# BWK Verhoudingen toevoegen
bwk_verhoudingen = ['1 op 2', '2 op 3', '3 op 4', '4 op 5']
for verhouding in bwk_verhoudingen:
    choices_list.append({
        "list_name": "lijst_bwk_verhoudingen",
        "name": utils.clean_name(verhouding),
        "label": verhouding
    })
    
df_choices_final = pd.DataFrame(choices_list)

# Zorg dat de talen overeenkomen in survey en choices!
if 'label' in df_choices_final.columns:
    df_choices_final.rename(columns={'label': 'label::nl'}, inplace=True)



Choices tabblad opbouwen...


In [474]:
df_choices_final[df_choices_final['list_name'] == 'lijst_habitats']

,list_name,name,label::nl
45,lijst_habitats,1310,1310
46,lijst_habitats,1320,1320
47,lijst_habitats,1330,1330
48,lijst_habitats,2110,2110
49,lijst_habitats,2120,2120
50,lijst_habitats,2130,2130
51,lijst_habitats,2150,2150
52,lijst_habitats,2160,2160
53,lijst_habitats,2170,2170
54,lijst_habitats,2180,2180


In [475]:
# Add settings dataframe
settings_df = pd.DataFrame([{
    "form_id": "lsvi_app_test",
    "form_title": "LSVI App Test",
    "style": "pages",   # <-- DIT ACTIVEERT HET GRID SYSTEEM IN DE APP
    "default_language": "nl" 
}])

**Waarschuwing: veel duplicate soorten in verschillende lijsten --> is dit normaal?**

## 2. Survey

### 2.1 Algemene info

In [476]:
print("Survey tabblad opbouwen...")
survey_list = []

# Unieke ID: Wordt op de achtergrond gegenereerd (gebruiker ziet dit niet)
survey_list.append({
    "type": "calculate", "name": "collectie_id", "label": "", 
    "calculation": "uuid()", "relevant": "", "appearance": "", "default": ""
})

# Datum & Uur: Automatisch ingevuld
survey_list.append({
    "type": "date", "name": "datum", "label": "Datum", 
    "default": "today()", "relevant": "", "appearance": "", "calculation": ""
})

survey_list.append({
    "type": "time", "name": "uur", "label": "Uur", 
    "default": "now()", "relevant": "", "appearance": "", "calculation": ""
})

Survey tabblad opbouwen...


### 2.2 BWK-vragen

**In finale versie kunnen velden op appearance: hidden gezet worden.**

In [477]:
# Maak grid aan voor hab and phab velden weer te geven
survey_list.append({
    "type": "begin group", 
    "name": "grp_fieldmaps_data", 
    "label": "Gegevens uit Field Maps (Controle)", 
    "relevant": "", 
    "appearance": "grid-layout"  # <-- Dit activeert het grid-systeem!
})

# Toon BWK ID over hele breedte
survey_list.append({
    "type": "integer", 
    "name": "bwk_id", 
    "label": "BWK ID", 
    "default": "", 
    "relevant": "", 
    "appearance": "w12", 
    "calculation": "", 
    "readonly": "yes"
})

# Toon HAB en PHAB in grid
for i in range(1, 6):
    # Linker kolom: Habitat Code (Breedte = w1)
    survey_list.append({
        "type": "text", 
        "name": f"hab{i}", 
        "label": f"Habitat {i}", 
        "default": "", 
        "relevant": "", 
        "appearance": "w6",  # <-- Neemt de helft van de ruimte in
        "calculation": "", 
        "readonly": "yes"
    })
    
    # Rechter kolom: Percentage (Breedte = w1)
    survey_list.append({
        "type": "integer", 
        "name": f"phab{i}", 
        "label": f"% Habitat {i}", 
        "default": "", 
        "relevant": "", 
        "appearance": "w6",  # <-- Neemt de andere helft van de ruimte in
        "calculation": "", 
        "readonly": "yes"
    })

# 4. Sluit de grid groep netjes af
survey_list.append({
    "type": "end group", "name": "", "label": "", "relevant": "", "appearance": ""
})

In [478]:
df_survey_final = pd.DataFrame(survey_list)

In [479]:
df_survey_final

,type,name,label,calculation,relevant,appearance,default,readonly
0,calculate,collectie_id,,uuid(),,,,NaN
1,date,datum,Datum,,,,today(),NaN
2,time,uur,Uur,,,,now(),NaN
3,begin group,grp_fieldmaps_data,Gegevens uit Field Maps (Controle),NaN,,grid-layout,NaN,NaN
4,integer,bwk_id,BWK ID,,,w12,,yes
5,text,hab1,Habitat 1,,,w6,,yes
6,integer,phab1,% Habitat 1,,,w6,,yes
7,text,hab2,Habitat 2,,,w6,,yes
8,integer,phab2,% Habitat 2,,,w6,,yes
9,text,hab3,Habitat 3,,,w6,,yes


### 2.3 Vragen per habitattype

In [480]:
# df_vereisten.AnalyseVariabele.unique()
# df_soorten[df_soorten['TaxongroepId'] == 285]
# df_vereisten[df_vereisten['Habitattype'] == 'rbbsg'][['Habitattype','Indicator','VoorwaardeID','Voorwaarde','AnalyseVariabele', 'Eenheid', 'TypeVariabele']]
df_vereisten[df_vereisten['VoorwaardeID'] == 2198]
# pd.DataFrame(survey_list).head()

,Versie,Habitattype,Habitatsubtype,Criterium,Indicator,Beoordeling,Kwaliteitsniveau,Belang,BeoordelingID,Combinatie,VoorwaardeID,Voorwaarde,Referentiewaarde,Operator,Maximumwaarde,AnalyseVariabele,Eenheid,TypeVariabele,Invoertype,Invoerwaarde,TaxongroepId,TaxongroepNaam,Studiegroepnaam,Studielijstnaam,Studiewaarde,SubAnalyseVariabele,SubEenheid,TypeSubVariabele,SubReferentiewaarde,SubOperator,SubInvoertype,SubInvoerwaarde,vraag_id,test_function
126,Versie 3,2120,2120,Structuur,horizontale structuur,fijnmazige afwisseling van naakte bodem met po...,1,b,2162,2198,2198,fijnmazige afwisseling,1,=,"1,00",meting,NaN,Ja/nee,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,vraag_2198_2120,2120
144,Versie 3,2120,2120_stuifduin,Structuur,horizontale structuur,fijnmazige afwisseling van naakte bodem met po...,1,b,2162,2198,2198,fijnmazige afwisseling,1,=,"1,00",meting,NaN,Ja/nee,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,vraag_2198_2120_stuifduin,2120_stuifduin
178,Versie 3,2120,2120_helmduin,Structuur,horizontale structuur,fijnmazige afwisseling van naakte bodem met po...,1,b,2162,2198,2198,fijnmazige afwisseling,1,=,"1,00",meting,NaN,Ja/nee,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,vraag_2198_2120_helmduin,2120_helmduin


In [481]:
# Trigger vraag
survey_list.append({
    "type": "select_one Ja_Nee", "name": "lsvi_opstellen", "label": "LSVI Opstellen?", 
    "relevant": "", "appearance": "horizontal", "default": "", "calculation": ""
})

# Groepeer alle LSVI vragen zodat we de 'relevant' logica maar 1 keer hoeven te typen
survey_list.append({
    "type": "begin group", "name": "grp_lsvi", "label": "LSVI Gegevens", 
    "relevant": "${lsvi_opstellen} = 'ja'", # Zichtbaar als vorige vraag 'ja' is
    "appearance": "field-list", "default": "", "calculation": ""
})

# Eerste hoofdvraag: Welk habitattype?
survey_list.append({
    "type": "select_multiple lijst_subhabitats",
    "name": "habitat_keuze",
    "label": "Welk habitat(sub)type wil je inventariseren?",
    "relevant": "",  # Altijd zichtbaar
    "hint": "Eenvoudige tekst om feature te testen.",
    "guidance_hint": "Eenvoudige tekst om feature te testen.",
    "appearance": "horizontal", #blank defaults to radio buttons instead of "minimal autocomplete",
    "choice_filter": "name = ${hab1} or name = ${hab2} or name = ${hab3} or name = ${hab4} or name = ${hab5}" # This makes sure we only get to choose habitats that were mapped in BWK field app for this polygon.
})

# 4.2. Loop door de unieke habitattypes (Creëer "Pages" / Groups)
for hab in unieke_subhabitats:
    hab_clean = utils.clean_name(hab)

    # Vragen per habitattype

    # Eerste repeat blok aanmaken zodat vragen in apparte tabel worden opgeslagen
    # Feature layer is beperkt tot 1024 kolommen (dus vragen)
    # Maken aparte tabel aan per habitattype om deze limiet te omzeilen
    survey_list.append({
        "type": "begin repeat",
        "name": f"rpt_habitat_{hab_clean}",  # 'rpt_' prefix helpt om database-tabellen te herkennen
        "label": "", # Keep label empty so it does not show in app
        "relevant": f"selected(${{habitat_keuze}}, '{hab_clean}')", # Toon tabel ENKEL als deze is aangevinkt
        "appearance": "minimal",  # Hakt de '+' en '-' knoppen weg voor de gebruiker
        "repeat_count": "1"        # Dwingt exact 1 gerelateerd record af (geen duplicaten mogelijk)
    })
    
    # Begin de groep voor dit specifieke habitattype. 
    # De "relevant" kolom vervangt de AppSheet Show_If: [LSVI opstellen voor] = 'rbbah'
    survey_list.append({
        "type": "begin group",
        "name": f"grp_habitat_{hab_clean}",
        "label": f"Habitat {hab.upper()}",
        "hint": utils.get_habitat_hint(hab),
        # "guidance_hint": get_habitat_hint(hab), # Dynamische hint genereren op basis van de habitattype code
        "relevant": "", #${{habitat_keuze}} = '{hab_clean}'", # De groep erft de relevantie van het repeat blok. Dit mag leeg zijn als we repeats gebruiken.
        "appearance": "compact field-list" # Zorgt dat het als 1 pagina toont in de app
    })

    # Filter de vereisten voor dít specifieke habitattype
    df_hab_vereisten = df_vereisten[df_vereisten['Habitatsubtype'] == hab]
    
    # 4.3. Genereer de vragen binnen dit habitattype
    for idx, row in df_hab_vereisten.iterrows():
        # Voorwaarde ID is enige unieke veld in invoervereisten
        # Joost belangrijk om te dubbelchecken
        vraag_naam = f"{row['vraag_id']}"
        
        # --- MAPPING LOGICA: WELKE SCHAAL GEBRUIKEN WE? ---
        # Dit is cruciaal: gebruik kolom AnalyseVariabele om te bepalen welk type vraag we moeten maken
        # Joost verifieer of dit klopt of TypeVariable ook gebruikt moet worden?
        type_var = str(row['AnalyseVariabele']).strip().lower()
        
        if 'bedekking' in type_var:
            vraag_type = "select_one Standaard"  # Verwijst naar de 'Standaard' lijst in survey123_schalen.csv
            vraag_appearance = "minimal autocomplete"  # Optioneel: maakt de opties naast elkaar in plaats van onder elkaar
        elif type_var == 'aantal' and pd.notna(row['TaxongroepId']): # als type vraag 'Aantal' is en Taxongroep is bekend kunnen we soortenlijst koppen aan multiple choice vraag
            tax_id = int(row['TaxongroepId'])
            vraag_type = f"select_multiple taxa_{tax_id}"
            vraag_appearance = "horizontal compact"
        elif type_var == 'meting_perc':
            vraag_type = "select_one Percentage" #to find example and implement joost
            vraag_appearance = "minimal"  
        else:
            vraag_type = "text" # Fallback
            vraag_appearance = "minimal" 
            print(f"Waarschuwing: Onbekend AnalyseVariabele type '{type_var}' voor VoorwaardeID {vraag_naam}. Fallback naar 'text' vraag.") 
            
        # Vraag toevoegen
        # Label van de vraag is combinatie van Voorwaarde + Indicator + Beoordeling(en eventueel Eenheid)
        # Add soortenlijst als hint 
        survey_list.append({
            "type": vraag_type,
            "name": vraag_naam,
            "label": utils.get_question_label(row), # De vraag die de gebruiker ziet
            # "hint": get_question_hint(row['TaxongroepId'], df_soorten),
            # "guidance_hint": get_question_hint(row['TaxongroepId'], df_soorten),
            "relevant": "",
            "appearance": vraag_appearance
        })

        # Genereer de soortenlijst HTML
        html_soorten = utils.get_question_hint(row['TaxongroepId'], df_soorten)

        # Check of er een soortenlijst is om te tonen
        if pd.notna(html_soorten) and str(html_soorten).strip() != "" and type_var == 'aantal':
            # Start de in- en uitklapbare groep
            survey_list.append({
                "type": "begin group",
                "name": f"grp_lijst_{vraag_naam}",
                "label": "🔽 Bekijk soortenlijst", # Dit is de tekst op de klikbare balk
                "hint": np.nan,
                "guidance_hint": np.nan,
                "relevant": "",
                "appearance": "compact" # <--- Dit commando maakt hem standaard ingeklapt!
            })

            # Voeg de 'note' toe met jouw HTML-geformatteerde soortenlijst
            survey_list.append({
                "type": "note",
                "name": f"not_{vraag_naam}",
                "label": html_soorten, # We stoppen jouw HTML nu in de 'label' van de note
                "hint": np.nan,
                "guidance_hint": np.nan,
                "relevant": "",
                "appearance": ""
            })

            # Sluit de uitklapbare groep netjes af
            survey_list.append({
                "type": "end group",
                "name": "", 
                "label": np.nan, 
                "hint": np.nan, 
                "guidance_hint": np.nan,
                "relevant": "", 
                "appearance": ""
            })

    # Sluit de groep (Pagina) af
    survey_list.append({
        "type": "end group",
        "name": "", "label": "", "relevant": "", "appearance": ""
    })

    survey_list.append({
        "type": "end repeat", "name": ""
    })

# Sluit de LSVI Groep af
survey_list.append({
    "type": "end group", "name": "", "label": "", 
    "relevant": "", "appearance": "", "default": "", "calculation": ""
})

df_survey_final = pd.DataFrame(survey_list)

Waarschuwing: Onbekend AnalyseVariabele type 'aandeelkruidlaag' voor VoorwaardeID vraag_93_1310_pol. Fallback naar 'text' vraag.
Waarschuwing: Onbekend AnalyseVariabele type 'meting' voor VoorwaardeID vraag_712_1310_zk. Fallback naar 'text' vraag.
Waarschuwing: Onbekend AnalyseVariabele type 'meting' voor VoorwaardeID vraag_2339_1310_zk. Fallback naar 'text' vraag.
Waarschuwing: Onbekend AnalyseVariabele type 'meting' voor VoorwaardeID vraag_2350_1310_zk. Fallback naar 'text' vraag.
Waarschuwing: Onbekend AnalyseVariabele type 'meting' voor VoorwaardeID vraag_2338_1310_zk. Fallback naar 'text' vraag.
Waarschuwing: Onbekend AnalyseVariabele type 'aandeelkruidlaag' voor VoorwaardeID vraag_96_1310_zk. Fallback naar 'text' vraag.
Waarschuwing: Onbekend AnalyseVariabele type 'meting' voor VoorwaardeID vraag_2430_1310_zv. Fallback naar 'text' vraag.
Waarschuwing: Onbekend AnalyseVariabele type 'meting' voor VoorwaardeID vraag_711_1320. Fallback naar 'text' vraag.
Waarschuwing: Onbekend Analy

### Add stresstest +1500 questions

In [482]:
# # Stresstest: add N additional questions to see if system (and feature layer in AGOL) has a limit

# n_additional_questions = 1200
# chunk_size = 400

# print(f"Generating {n_additional_questions} questions spread across multiple related tables...")

# for chunk_start in range(0, n_additional_questions, chunk_size):
#     chunk_num = (chunk_start // chunk_size) + 1

#     print('chunk num is ', chunk_num)
    
#     # 1. Start a new repeat (which creates a brand new database table)
#     survey_list.append({
#         "type": "begin repeat", 
#         "name": f"rpt_stress_table_{chunk_num}", 
#         "label": f"Stresstest Table {chunk_num}", 
#         "relevant": "", 
#         "appearance": "minimal",  # Hides the UI addition buttons
#         "repeat_count": "1"        # Seamlessly forces exactly 1 related record
#     })

#     print('chunk start is ', chunk_start)
    
#     # 2. Fill this specific table with questions up to the CHUNK_SIZE
#     chunk_end = min(chunk_start + chunk_size, n_additional_questions)
#     for i in range(chunk_start + 1, chunk_end + 1):
#         survey_list.append({
#             "type": "text", 
#             "name": f"stress_test_col_{i}", 
#             "label": f"Stresstest Vraag {i}", 
#             "relevant": "", 
#             "appearance": "minimal"
#         })
        
#     # 3. Close this repeat table
#     survey_list.append({
#         "type": "end repeat", "name": "", "label": "", "relevant": "", "appearance": ""
#     })

# df_survey_final = pd.DataFrame(survey_list)


### 2.4 Export to XLSX

In [483]:
import numpy as np
# Vervang alle lege strings in de hint (en eventueel andere) kolommen door echte NaN waarden
df_survey_final['hint'] = df_survey_final['hint'].replace("", np.nan)
# df_survey_final['guidance_hint'] = df_survey_final['guidance_hint'].replace("", np.nan)

# rename columns to add default language
df_survey_final = df_survey_final.rename(columns={'label': 'label::nl', 'hint': 'hint::nl', 'guidance_hint': 'guidance_hint::nl'})

print("Excel bestand genereren...")
output_file = '../output/Survey123_Volledige_Generatie_20260611.xlsx'
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_survey_final.to_excel(writer, sheet_name='survey', index=False)
    df_choices_final.to_excel(writer, sheet_name='choices', index=False)
    # Voeg een standaard settings tabblad toe
    pd.DataFrame([{"form_title": "Habitattype Survey", "form_id": "habitat_survey_v1", "default_language": "nl"}]).to_excel(writer, sheet_name='settings', index=False)

# Overwrite Survey123 test app
# output_file = r'C:\Users\joost_neujens\ArcGIS\My Survey Designs\LSVI App Test\LSVI App Test.xlsx' # if running local
output_file = r"\\Client\C$\Users\joost_neujens\ArcGIS\My Survey Designs\LSVI App Test\LSVI App Test.xlsx"
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_survey_final.to_excel(writer, sheet_name='survey', index=False)
    df_choices_final.to_excel(writer, sheet_name='choices', index=False)
    # Voeg een standaard settings tabblad toe
    settings_df.to_excel(writer, sheet_name='settings', index=False)

print(f"✅ Survey123 XLSForm succesvol gegenereerd: '{output_file}'")

Excel bestand genereren...
✅ Survey123 XLSForm succesvol gegenereerd: '\\Client\C$\Users\joost_neujens\ArcGIS\My Survey Designs\LSVI App Test\LSVI App Test.xlsx'


In [485]:
df_survey_final.shape

(1449, 12)

In [484]:
df_survey_final.head(30)

,type,name,label::nl,calculation,relevant,appearance,default,readonly,hint::nl,guidance_hint::nl,choice_filter,repeat_count
0,calculate,collectie_id,,uuid(),,,,NaN,NaN,NaN,NaN,NaN
1,date,datum,Datum,,,,today(),NaN,NaN,NaN,NaN,NaN
2,time,uur,Uur,,,,now(),NaN,NaN,NaN,NaN,NaN
3,begin group,grp_fieldmaps_data,Gegevens uit Field Maps (Controle),NaN,,grid-layout,NaN,NaN,NaN,NaN,NaN,NaN
4,integer,bwk_id,BWK ID,,,w12,,yes,NaN,NaN,NaN,NaN
5,text,hab1,Habitat 1,,,w6,,yes,NaN,NaN,NaN,NaN
6,integer,phab1,% Habitat 1,,,w6,,yes,NaN,NaN,NaN,NaN
7,text,hab2,Habitat 2,,,w6,,yes,NaN,NaN,NaN,NaN
8,integer,phab2,% Habitat 2,,,w6,,yes,NaN,NaN,NaN,NaN
9,text,hab3,Habitat 3,,,w6,,yes,NaN,NaN,NaN,NaN
